# 04 RoBERTa Pretraining

## Purpose

This notebook runs masked language model pretraining for one tokenizer setting and one model configuration.

## Inputs

- tokenizer files from one folder in `MyDrive/ProjectRoot/tokenizers/`
- tokenized datasets from one folder in `MyDrive/ProjectRoot/tokenized_datasets/`
- optional checkpoint or `best_model` folder for continuation runs

## Outputs

- training checkpoints in `MyDrive/ProjectRoot/checkpoints/<tokenizer_family>/<experiment_name>/`
- `best_model/` saved inside that experiment folder
- `trainer_state.json`
- `experiment_metadata.json`
- run index updates written to `MyDrive/ProjectRoot/registry/run_index.csv`

## Notes to myself

This is the main training notebook, so I want it to stay explicit. The two biggest things are making sure the run naming is clean and making sure continuation runs don't quietly point at the wrong tokenizer or dataset.

## Setup note

Same pattern again.

- code and notebooks stay in GitHub
- checkpoints and heavy training artifacts stay in Drive
- Colab pulls the repo at the start
- the final cell syncs the notebook back to GitHub

In [24]:
# ==============================================================================
# 0. SET UP THE COLAB ENVIRONMENT
# ==============================================================================
import os
import sys

from google.colab import drive

# Mount Google Drive so the notebook can read data files and save outputs.
drive.mount('/content/drive')

# Force tqdm to use plain text output instead of notebook widgets. This keeps
# GitHub preview from breaking when I save the notebook back from Colab.
from tqdm.std import tqdm as plain_tqdm
import tqdm.auto as tqdm_auto
tqdm_auto.tqdm = plain_tqdm
try:
    import tqdm.notebook as tqdm_notebook
    tqdm_notebook.tqdm = plain_tqdm
except Exception:
    pass

# This repository is public, so Colab can clone it without authentication.
GITHUB_OWNER = 'hb791-dev'
REPO_NAME = 'glycan-roberta'
REPO_URL = f'https://github.com/{GITHUB_OWNER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    print('Cloning repository...')
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    print('Repository already exists. Pulling latest changes...')

%cd {REPO_DIR}
!git pull origin main --no-edit -q

# Add the repo to the Python path so src/ imports work across notebooks.
if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

print('Colab environment ready.')
print(f'Repo directory: {REPO_DIR}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repository already exists. Pulling latest changes...
/content/glycan-roberta
Colab environment ready.
Repo directory: /content/glycan-roberta


## Run modes

This notebook supports three modes:

- `fresh`: start a brand-new training run
- `resume_checkpoint`: continue from a saved `checkpoint-*` folder
- `continue_best_model`: start a new continuation run from a saved `best_model` folder

The main thing to remember is:

- `resume_checkpoint` keeps the original learning-rate plan and expects total target epochs
- `continue_best_model` is a new experiment and expects only the extra continuation length

In [25]:
# ==============================================================================
# 1. DEFINE THE TRAINING CONFIGURATION
# ==============================================================================
import json
import subprocess

# --- A. RUN MODE CONTROL ---
RUN_MODE = 'fresh'
# 'fresh', 'resume_checkpoint', or 'continue_best_model'

PARENT_EXPERIMENT_NAME = None
RESUME_SOURCE_DIR = None
# These stay empty for fresh runs. Fill them in only for continuation modes.
# Example checkpoint path:
# '/content/drive/MyDrive/ProjectRoot/checkpoints/byte_bpe/mlm15_L6_H512_A8_lr00001_ep100_setv300_m2/checkpoint-54600'
# Example best_model path:
# '/content/drive/MyDrive/ProjectRoot/checkpoints/byte_bpe/mlm15_L6_H512_A8_lr00001_ep100_setv300_m2/best_model'

# --- B. TOKENIZER AND DATASET SETTINGS ---
TOKENIZER_FAMILY = 'hybrid_char_bpe'   # 'byte_bpe', 'manual', or 'hybrid_char_bpe'
SETTING_LABEL = 'v70_m2'               # examples: 'v300_m2', 'v1_train_only', 'v70_m2'
MLM_PROBABILITY = 0.15

# --- C. MODEL SETTINGS ---
NUM_HIDDEN_LAYERS = 8
ATTENTION_HEADS = 8
HIDDEN_SIZE = 512
INTERMEDIATE_SIZE = HIDDEN_SIZE * 4
MAX_POSITION_EMBEDDINGS = 512

# --- D. TRAINING SETTINGS ---
BATCH_SIZE = 32
WEIGHT_DECAY = 0.01
SAVE_TOTAL_LIMIT = 3
EARLY_STOPPING_PATIENCE = 15
LOGGING_STEPS = 50
RANDOM_SEED = 42

# --- E. TRAINING LENGTH AND LEARNING RATE ---
INITIAL_EPOCHS = 100
CONTINUATION_EPOCHS = 20

BASE_LEARNING_RATE = 1e-4
CONTINUATION_LEARNING_RATE = 5e-5

# Convert the run mode into the effective training length and learning rate.
if RUN_MODE == 'fresh':
    EPOCHS = INITIAL_EPOCHS
    LEARNING_RATE = BASE_LEARNING_RATE
elif RUN_MODE == 'resume_checkpoint':
    if not PARENT_EXPERIMENT_NAME or not RESUME_SOURCE_DIR:
        raise ValueError('resume_checkpoint mode requires PARENT_EXPERIMENT_NAME and RESUME_SOURCE_DIR')
    EPOCHS = INITIAL_EPOCHS + CONTINUATION_EPOCHS
    LEARNING_RATE = BASE_LEARNING_RATE
elif RUN_MODE == 'continue_best_model':
    if not PARENT_EXPERIMENT_NAME or not RESUME_SOURCE_DIR:
        raise ValueError('continue_best_model mode requires PARENT_EXPERIMENT_NAME and RESUME_SOURCE_DIR')
    EPOCHS = CONTINUATION_EPOCHS
    LEARNING_RATE = CONTINUATION_LEARNING_RATE
else:
    raise ValueError(f'Unsupported RUN_MODE: {RUN_MODE}')

# Build the main Drive paths used by this run.
PROJECT_ROOT = '/content/drive/MyDrive/ProjectRoot'
CHECKPOINT_ROOT = os.path.join(PROJECT_ROOT, 'checkpoints', TOKENIZER_FAMILY)
TOKENIZER_DIR = os.path.join(PROJECT_ROOT, 'tokenizers', TOKENIZER_FAMILY, SETTING_LABEL)
TOKENIZED_DATASET_DIR = os.path.join(PROJECT_ROOT, 'tokenized_datasets', TOKENIZER_FAMILY, SETTING_LABEL)
RUN_INDEX_PATH = os.path.join(PROJECT_ROOT, 'registry', 'run_index.csv')

# Make sure the tokenizer and tokenized datasets already exist before training.
for required_path in [TOKENIZER_DIR, TOKENIZED_DATASET_DIR]:
    if not os.path.exists(required_path):
        raise FileNotFoundError(f'Required path not found: {required_path}')

# Check that continuation modes point at the right kind of saved directory.
if RUN_MODE == 'resume_checkpoint':
    if not os.path.exists(RESUME_SOURCE_DIR):
        raise FileNotFoundError(f'Checkpoint not found: {RESUME_SOURCE_DIR}')
    if 'checkpoint-' not in os.path.basename(RESUME_SOURCE_DIR):
        raise ValueError('resume_checkpoint mode must point to a checkpoint-* directory')

if RUN_MODE == 'continue_best_model':
    if not os.path.exists(RESUME_SOURCE_DIR):
        raise FileNotFoundError(f'best_model directory not found: {RESUME_SOURCE_DIR}')
    if os.path.basename(RESUME_SOURCE_DIR) != 'best_model':
        raise ValueError('continue_best_model mode must point to a best_model directory')

# For continuation runs, check that the saved model architecture matches what
# this notebook is about to request.
if RUN_MODE in ['resume_checkpoint', 'continue_best_model']:
    resume_config_path = os.path.join(RESUME_SOURCE_DIR, 'config.json')
    if os.path.exists(resume_config_path):
        with open(resume_config_path, 'r', encoding='utf-8') as file:
            resume_config = json.load(file)

        expected_pairs = {
            'num_hidden_layers': NUM_HIDDEN_LAYERS,
            'num_attention_heads': ATTENTION_HEADS,
            'hidden_size': HIDDEN_SIZE,
            'intermediate_size': INTERMEDIATE_SIZE,
            'vocab_size': None,
        }

        for key, expected in expected_pairs.items():
            if key == 'vocab_size':
                continue
            observed = resume_config.get(key)
            if observed != expected:
                raise ValueError(f'Resume model mismatch for {key}: expected {expected}, found {observed}')

# Save the exact repo commit used for this run in the experiment metadata.
git_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR).decode('utf-8').strip()

print('Configuration loaded.')
print(f'Run mode: {RUN_MODE}')
print(f'Tokenizer family: {TOKENIZER_FAMILY}')
print(f'Setting label: {SETTING_LABEL}')
print(f'Learning rate: {LEARNING_RATE}')
print(f'Epochs: {EPOCHS}')

Configuration loaded.
Run mode: fresh
Tokenizer family: hybrid_char_bpe
Setting label: v70_m2
Learning rate: 0.0001
Epochs: 100


## Run naming and metadata

I want the experiment folder name to carry the key training settings directly. I also want every run to register itself right away so the run index doesn't depend on me remembering to document it later.

In [26]:
# ==============================================================================
# 2. BUILD THE EXPERIMENT NAME AND REGISTER THE RUN
# ==============================================================================
from src.run_index import upsert_run_record

def format_lr_tag(value):
    return str(value).replace('.', '')

def build_base_experiment_name():
    arch_tag = f'L{NUM_HIDDEN_LAYERS}_H{HIDDEN_SIZE}_A{ATTENTION_HEADS}'
    lr_tag = format_lr_tag(LEARNING_RATE)

    # Fresh runs name the new architecture directly. Continuation modes keep
    # the parent experiment in the new folder name.
    if RUN_MODE == 'fresh':
        return f'mlm{int(MLM_PROBABILITY * 100)}_{arch_tag}_lr{lr_tag}_ep{EPOCHS}_set{SETTING_LABEL}'
    if RUN_MODE == 'resume_checkpoint':
        return f'{PARENT_EXPERIMENT_NAME}_resume_toep{EPOCHS}'
    return f'{PARENT_EXPERIMENT_NAME}_cont_lr{lr_tag}_ep{EPOCHS}'

def resolve_experiment_dir(base_dir):
    # If a folder name already exists, make a versioned copy instead of
    # overwriting an older run.
    if not os.path.exists(base_dir):
        return base_dir

    version = 2
    while True:
        candidate = f'{base_dir}_v{version}'
        if not os.path.exists(candidate):
            return candidate
        version += 1

BASE_EXPERIMENT_NAME = build_base_experiment_name()
BASE_CHECKPOINT_DIR = os.path.join(CHECKPOINT_ROOT, BASE_EXPERIMENT_NAME)
CHECKPOINT_DIR = resolve_experiment_dir(BASE_CHECKPOINT_DIR)
EXPERIMENT_NAME = os.path.basename(CHECKPOINT_DIR)
BEST_MODEL_DIR = os.path.join(CHECKPOINT_DIR, 'best_model')
TRAINER_STATE_PATH = os.path.join(CHECKPOINT_DIR, 'trainer_state.json')
LOG_DIR = os.path.join(CHECKPOINT_DIR, 'logs')
EXPERIMENT_METADATA_PATH = os.path.join(CHECKPOINT_DIR, 'experiment_metadata.json')

# Create the run folder before writing metadata or training artifacts.
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

# Write a first-pass metadata file before training starts.
metadata_payload = {
    'experiment_name': EXPERIMENT_NAME,
    'notebook_used': 'notebooks/04_roberta_pretraining.ipynb',
    'git_commit': git_commit,
    'vault_routing': {
        'tokenizer_dir': TOKENIZER_DIR,
        'tokenized_dataset_dir': TOKENIZED_DATASET_DIR,
        'checkpoint_dir': CHECKPOINT_DIR,
        'best_model_dir': BEST_MODEL_DIR,
        'run_index_path': RUN_INDEX_PATH,
    },
    'live_hyperparameters': {
        'run_mode': RUN_MODE,
        'parent_experiment_name': PARENT_EXPERIMENT_NAME,
        'resume_source_dir': RESUME_SOURCE_DIR,
        'tokenizer_family': TOKENIZER_FAMILY,
        'setting_label': SETTING_LABEL,
        'mlm_probability': MLM_PROBABILITY,
        'num_hidden_layers': NUM_HIDDEN_LAYERS,
        'attention_heads': ATTENTION_HEADS,
        'hidden_size': HIDDEN_SIZE,
        'intermediate_size': INTERMEDIATE_SIZE,
        'max_position_embeddings': MAX_POSITION_EMBEDDINGS,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'epochs': EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'save_total_limit': SAVE_TOTAL_LIMIT,
        'random_seed': RANDOM_SEED,
        'initial_epochs': INITIAL_EPOCHS,
        'continuation_epochs': CONTINUATION_EPOCHS,
        'base_learning_rate': BASE_LEARNING_RATE,
        'continuation_learning_rate': CONTINUATION_LEARNING_RATE,
    },
    'run_status': 'configured',
}

with open(EXPERIMENT_METADATA_PATH, 'w', encoding='utf-8') as file:
    json.dump(metadata_payload, file, indent=2)

# Register the run immediately so the index records configured runs too.
upsert_run_record(
    RUN_INDEX_PATH,
    {
        'experiment_name': EXPERIMENT_NAME,
        'tokenizer_family': TOKENIZER_FAMILY,
        'setting_label': SETTING_LABEL,
        'run_mode': RUN_MODE,
        'parent_experiment_name': PARENT_EXPERIMENT_NAME,
        'mlm_probability': MLM_PROBABILITY,
        'num_hidden_layers': NUM_HIDDEN_LAYERS,
        'attention_heads': ATTENTION_HEADS,
        'hidden_size': HIDDEN_SIZE,
        'intermediate_size': INTERMEDIATE_SIZE,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'epochs': EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'tokenizer_dir': TOKENIZER_DIR,
        'tokenized_dataset_dir': TOKENIZED_DATASET_DIR,
        'checkpoint_dir': CHECKPOINT_DIR,
        'results_dir': CHECKPOINT_DIR,
        'notebook_used': 'notebooks/04_roberta_pretraining.ipynb',
        'git_commit': git_commit,
        'run_status': 'configured',
        'notes': '',
    },
)

print(f'Experiment name: {EXPERIMENT_NAME}')
print(f'Checkpoint directory: {CHECKPOINT_DIR}')
print(f'Run index path: {RUN_INDEX_PATH}')

Experiment name: mlm15_L8_H512_A8_lr00001_ep100_setv70_m2
Checkpoint directory: /content/drive/MyDrive/ProjectRoot/checkpoints/hybrid_char_bpe/mlm15_L8_H512_A8_lr00001_ep100_setv70_m2
Run index path: /content/drive/MyDrive/ProjectRoot/registry/run_index.csv


## Load the tokenizer

I want this separate from the config cell because it gives me a clean place to verify the vocabulary and special tokens before training starts.

In [27]:
# ==============================================================================
# 3. LOAD THE TOKENIZER
# ==============================================================================
from transformers import PreTrainedTokenizerFast

# Load the tokenizer exactly as it was saved in notebook 02.
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    TOKENIZER_DIR,
    bos_token='<s>',
    eos_token='</s>',
    unk_token='<unk>',
    pad_token='<pad>',
    mask_token='<mask>'
)

VOCAB_SIZE = len(tokenizer)
PAD_TOKEN_ID = tokenizer.pad_token_id
MASK_TOKEN_ID = tokenizer.mask_token_id

print(f'Tokenizer loaded from: {TOKENIZER_DIR}')
print(f'Vocabulary size: {VOCAB_SIZE}')
print(f'Pad token ID: {PAD_TOKEN_ID}')
print(f'Mask token ID: {MASK_TOKEN_ID}')

Tokenizer loaded from: /content/drive/MyDrive/ProjectRoot/tokenizers/hybrid_char_bpe/v70_m2
Vocabulary size: 70
Pad token ID: 1
Mask token ID: 4


## Load the tokenized datasets

This notebook should only touch the train and validation splits. The test tensors should already exist from notebook 3, but they are for notebook 6, not for training decisions here.

In [28]:
# ==============================================================================
# 4. LOAD THE TOKENIZED TRAIN AND VALIDATION DATASETS
# ==============================================================================
import torch
from torch.utils.data import Dataset

# Wrap the saved tensor dictionaries so Hugging Face Trainer can iterate over them.
class GlycanDataset(Dataset):
    def __init__(self, dataset_dict):
        self.input_ids = dataset_dict['input_ids']
        self.attention_mask = dataset_dict['attention_mask']

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
        }

train_path = os.path.join(TOKENIZED_DATASET_DIR, 'train_dataset.pt')
val_path = os.path.join(TOKENIZED_DATASET_DIR, 'val_dataset.pt')
summary_path = os.path.join(TOKENIZED_DATASET_DIR, 'preprocessing_summary.json')

# Notebook 04 should only use train and validation tensors.
for required_path in [train_path, val_path]:
    if not os.path.exists(required_path):
        raise FileNotFoundError(f'Preprocessed dataset not found: {required_path}')

# Load the tokenized splits created in notebook 03.
raw_train = torch.load(train_path)
raw_val = torch.load(val_path)

train_dataset = GlycanDataset(raw_train)
val_dataset = GlycanDataset(raw_val)

train_sequence_width = int(train_dataset.input_ids.shape[1])
# Catch mismatches between tokenizer preprocessing length and model position limit.
if train_sequence_width > MAX_POSITION_EMBEDDINGS:
    raise ValueError(
        f'MAX_POSITION_EMBEDDINGS={MAX_POSITION_EMBEDDINGS} is smaller than tokenized sequence width {train_sequence_width}'
    )

preprocessing_summary = {}
if os.path.exists(summary_path):
    with open(summary_path, 'r', encoding='utf-8') as file:
        preprocessing_summary = json.load(file)

print(f'Train dataset size: {len(train_dataset)}')
print(f'Validation dataset size: {len(val_dataset)}')
print(f'Sequence width: {train_sequence_width}')
if preprocessing_summary:
    print(f"Selected max length from notebook 03: {preprocessing_summary.get('selected_max_length', 'not found')}")

Train dataset size: 17453
Validation dataset size: 2182
Sequence width: 56
Selected max length from notebook 03: 56


## Initialize the model

Fresh and resume-checkpoint runs start from the declared config. `continue_best_model` loads the saved best weights directly because that mode is meant to start a new experiment from a previously trained model.

In [29]:
# ==============================================================================
# 5. INITIALIZE THE MODEL
# ==============================================================================
from transformers import RobertaConfig, RobertaForMaskedLM

# Define the transformer architecture for fresh runs or checkpoint resumes.
config = RobertaConfig(
    vocab_size=VOCAB_SIZE,
    max_position_embeddings=MAX_POSITION_EMBEDDINGS,
    num_hidden_layers=NUM_HIDDEN_LAYERS,
    num_attention_heads=ATTENTION_HEADS,
    hidden_size=HIDDEN_SIZE,
    intermediate_size=INTERMEDIATE_SIZE,
    pad_token_id=PAD_TOKEN_ID,
    type_vocab_size=1,
)

# Fresh and resume-checkpoint runs start from this declared config. Planned
# continuation runs start from a saved best_model folder instead.
if RUN_MODE in ['fresh', 'resume_checkpoint']:
    model = RobertaForMaskedLM(config)
elif RUN_MODE == 'continue_best_model':
    print(f'Loading best model weights from: {RESUME_SOURCE_DIR}')
    model = RobertaForMaskedLM.from_pretrained(RESUME_SOURCE_DIR)
else:
    raise ValueError(f'Unsupported RUN_MODE: {RUN_MODE}')

total_trainable_params = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

metadata_payload['model_summary'] = {
    'total_trainable_parameters': int(total_trainable_params),
    'vocab_size': int(VOCAB_SIZE),
    'sequence_width': int(train_sequence_width),
}

with open(EXPERIMENT_METADATA_PATH, 'w', encoding='utf-8') as file:
    json.dump(metadata_payload, file, indent=2)

print(f'Total trainable parameters: {total_trainable_params:,}')

Total trainable parameters: 25,782,342


## Configure training

This cell is where the masking collator and the Hugging Face training arguments get locked in. I want those choices saved into the experiment folder through the metadata file and trainer state.

In [30]:
# ==============================================================================
# 6. CONFIGURE THE DATA COLLATOR AND TRAINING ARGUMENTS
# ==============================================================================
from transformers import DataCollatorForLanguageModeling, TrainingArguments


# Apply random masking on the fly during MLM training.
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=MLM_PROBABILITY,
)

# Use epoch-level evaluation and checkpointing so later diagnostics line up with epochs.
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    logging_steps=LOGGING_STEPS,
    disable_tqdm=True,
    report_to='none',
    seed=RANDOM_SEED,
    data_seed=RANDOM_SEED,
    # Use mixed precision automatically when a CUDA GPU is available.
    fp16=torch.cuda.is_available(),
)

print(f'Training outputs will be saved to: {CHECKPOINT_DIR}')
print(f'fp16 enabled: {torch.cuda.is_available()}')


Training outputs will be saved to: /content/drive/MyDrive/ProjectRoot/checkpoints/hybrid_char_bpe/mlm15_L8_H512_A8_lr00001_ep100_setv70_m2
fp16 enabled: True


## Run training

This is the actual MLM training step. Right before it starts, I mark the run as `running` in the index. When it finishes, I save the best model, save the trainer state, and mark the run as `completed`.

In [31]:
# ==============================================================================
# 7. RUN MLM PRETRAINING
# ==============================================================================
from transformers import EarlyStoppingCallback, Trainer
from tqdm.std import tqdm as plain_tqdm

# Patch transformers save-time progress bars so they stay plain text in Colab.
try:
    import transformers.modeling_utils as modeling_utils
    modeling_utils.tqdm = plain_tqdm
except Exception:
    pass

try:
    import transformers.trainer as trainer_module
    trainer_module.tqdm = plain_tqdm
except Exception:
    pass

# Mark the run as active before the trainer starts.
metadata_payload['run_status'] = 'running'
with open(EXPERIMENT_METADATA_PATH, 'w', encoding='utf-8') as file:
    json.dump(metadata_payload, file, indent=2)

upsert_run_record(
    RUN_INDEX_PATH,
    {
        'experiment_name': EXPERIMENT_NAME,
        'tokenizer_family': TOKENIZER_FAMILY,
        'setting_label': SETTING_LABEL,
        'run_mode': RUN_MODE,
        'parent_experiment_name': PARENT_EXPERIMENT_NAME,
        'mlm_probability': MLM_PROBABILITY,
        'num_hidden_layers': NUM_HIDDEN_LAYERS,
        'attention_heads': ATTENTION_HEADS,
        'hidden_size': HIDDEN_SIZE,
        'intermediate_size': INTERMEDIATE_SIZE,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'epochs': EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'tokenizer_dir': TOKENIZER_DIR,
        'tokenized_dataset_dir': TOKENIZED_DATASET_DIR,
        'checkpoint_dir': CHECKPOINT_DIR,
        'results_dir': CHECKPOINT_DIR,
        'notebook_used': 'notebooks/04_roberta_pretraining.ipynb',
        'git_commit': git_commit,
        'run_status': 'running',
        'notes': '',
    },
)

# The Trainer handles MLM masking, checkpoint saving, and validation evaluation.
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

train_kwargs = {}
# Only checkpoint resumes should restore trainer state directly.
if RUN_MODE == 'resume_checkpoint':
    print(f'Resuming trainer state from checkpoint: {RESUME_SOURCE_DIR}')
    train_kwargs['resume_from_checkpoint'] = RESUME_SOURCE_DIR

# Start training and then save the selected best model into a stable folder.
trainer.train(**train_kwargs)
trainer.save_state()
trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

# Final metadata and run-index update after successful completion.
metadata_payload['run_status'] = 'completed'
metadata_payload['training_artifacts'] = {
    'trainer_state_path': TRAINER_STATE_PATH,
    'best_model_dir': BEST_MODEL_DIR,
    'log_dir': LOG_DIR,
}

with open(EXPERIMENT_METADATA_PATH, 'w', encoding='utf-8') as file:
    json.dump(metadata_payload, file, indent=2)

upsert_run_record(
    RUN_INDEX_PATH,
    {
        'experiment_name': EXPERIMENT_NAME,
        'tokenizer_family': TOKENIZER_FAMILY,
        'setting_label': SETTING_LABEL,
        'run_mode': RUN_MODE,
        'parent_experiment_name': PARENT_EXPERIMENT_NAME,
        'mlm_probability': MLM_PROBABILITY,
        'num_hidden_layers': NUM_HIDDEN_LAYERS,
        'attention_heads': ATTENTION_HEADS,
        'hidden_size': HIDDEN_SIZE,
        'intermediate_size': INTERMEDIATE_SIZE,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'epochs': EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'tokenizer_dir': TOKENIZER_DIR,
        'tokenized_dataset_dir': TOKENIZED_DATASET_DIR,
        'checkpoint_dir': CHECKPOINT_DIR,
        'results_dir': CHECKPOINT_DIR,
        'notebook_used': 'notebooks/04_roberta_pretraining.ipynb',
        'git_commit': git_commit,
        'run_status': 'completed',
        'notes': '',
    },
)

print('Training complete.')
print(f'Best model saved to: {BEST_MODEL_DIR}')
print(f'Trainer state saved to: {TRAINER_STATE_PATH}')


{'loss': '3.291', 'grad_norm': '3.662', 'learning_rate': '9.991e-05', 'epoch': '0.09158'}
{'loss': '2.76', 'grad_norm': '4.533', 'learning_rate': '9.982e-05', 'epoch': '0.1832'}
{'loss': '2.579', 'grad_norm': '3.79', 'learning_rate': '9.973e-05', 'epoch': '0.2747'}
{'loss': '2.462', 'grad_norm': '5.083', 'learning_rate': '9.964e-05', 'epoch': '0.3663'}
{'loss': '2.318', 'grad_norm': '5.015', 'learning_rate': '9.954e-05', 'epoch': '0.4579'}
{'loss': '2.224', 'grad_norm': '4.287', 'learning_rate': '9.945e-05', 'epoch': '0.5495'}
{'loss': '1.942', 'grad_norm': '5.833', 'learning_rate': '9.936e-05', 'epoch': '0.641'}
{'loss': '1.821', 'grad_norm': '5.583', 'learning_rate': '9.927e-05', 'epoch': '0.7326'}
{'loss': '1.639', 'grad_norm': '6.932', 'learning_rate': '9.918e-05', 'epoch': '0.8242'}
{'loss': '1.423', 'grad_norm': '5.277', 'learning_rate': '9.909e-05', 'epoch': '0.9158'}
{'eval_loss': '1.092', 'eval_runtime': '0.9367', 'eval_samples_per_second': '2329', 'eval_steps_per_second': '73

{'loss': '1.29', 'grad_norm': '5.419', 'learning_rate': '9.899e-05', 'epoch': '1.007'}
{'loss': '1.08', 'grad_norm': '6.189', 'learning_rate': '9.89e-05', 'epoch': '1.099'}
{'loss': '0.9872', 'grad_norm': '4.321', 'learning_rate': '9.881e-05', 'epoch': '1.19'}
{'loss': '0.8817', 'grad_norm': '6.038', 'learning_rate': '9.872e-05', 'epoch': '1.282'}
{'loss': '0.8079', 'grad_norm': '3.73', 'learning_rate': '9.863e-05', 'epoch': '1.374'}
{'loss': '0.7536', 'grad_norm': '4.551', 'learning_rate': '9.854e-05', 'epoch': '1.465'}
{'loss': '0.6971', 'grad_norm': '5.151', 'learning_rate': '9.845e-05', 'epoch': '1.557'}
{'loss': '0.6406', 'grad_norm': '4.179', 'learning_rate': '9.835e-05', 'epoch': '1.648'}
{'loss': '0.6212', 'grad_norm': '2.901', 'learning_rate': '9.826e-05', 'epoch': '1.74'}
{'loss': '0.6049', 'grad_norm': '3.608', 'learning_rate': '9.817e-05', 'epoch': '1.832'}
{'loss': '0.6033', 'grad_norm': '3.364', 'learning_rate': '9.808e-05', 'epoch': '1.923'}
{'eval_loss': '0.5185', 'eval

{'loss': '0.5727', 'grad_norm': '3.017', 'learning_rate': '9.799e-05', 'epoch': '2.015'}
{'loss': '0.5438', 'grad_norm': '3.596', 'learning_rate': '9.79e-05', 'epoch': '2.106'}
{'loss': '0.5291', 'grad_norm': '3.061', 'learning_rate': '9.78e-05', 'epoch': '2.198'}
{'loss': '0.4907', 'grad_norm': '2.455', 'learning_rate': '9.771e-05', 'epoch': '2.289'}
{'loss': '0.5119', 'grad_norm': '4.643', 'learning_rate': '9.762e-05', 'epoch': '2.381'}
{'loss': '0.5172', 'grad_norm': '3.705', 'learning_rate': '9.753e-05', 'epoch': '2.473'}
{'loss': '0.5127', 'grad_norm': '4.414', 'learning_rate': '9.744e-05', 'epoch': '2.564'}
{'loss': '0.4474', 'grad_norm': '2.88', 'learning_rate': '9.735e-05', 'epoch': '2.656'}
{'loss': '0.4466', 'grad_norm': '3.91', 'learning_rate': '9.725e-05', 'epoch': '2.747'}
{'loss': '0.4602', 'grad_norm': '4.384', 'learning_rate': '9.716e-05', 'epoch': '2.839'}
{'loss': '0.4259', 'grad_norm': '3.783', 'learning_rate': '9.707e-05', 'epoch': '2.93'}
{'eval_loss': '0.4003', 'e

{'loss': '0.4485', 'grad_norm': '3.565', 'learning_rate': '9.698e-05', 'epoch': '3.022'}
{'loss': '0.4313', 'grad_norm': '1.601', 'learning_rate': '9.689e-05', 'epoch': '3.114'}
{'loss': '0.4709', 'grad_norm': '3.049', 'learning_rate': '9.68e-05', 'epoch': '3.205'}
{'loss': '0.4122', 'grad_norm': '3.587', 'learning_rate': '9.671e-05', 'epoch': '3.297'}
{'loss': '0.3899', 'grad_norm': '2.07', 'learning_rate': '9.661e-05', 'epoch': '3.388'}
{'loss': '0.4206', 'grad_norm': '3.994', 'learning_rate': '9.652e-05', 'epoch': '3.48'}
{'loss': '0.4416', 'grad_norm': '3.837', 'learning_rate': '9.643e-05', 'epoch': '3.571'}
{'loss': '0.4096', 'grad_norm': '2.408', 'learning_rate': '9.634e-05', 'epoch': '3.663'}
{'loss': '0.4236', 'grad_norm': '5.102', 'learning_rate': '9.625e-05', 'epoch': '3.755'}
{'loss': '0.3666', 'grad_norm': '3.047', 'learning_rate': '9.616e-05', 'epoch': '3.846'}
{'loss': '0.4031', 'grad_norm': '4.065', 'learning_rate': '9.606e-05', 'epoch': '3.938'}
{'eval_loss': '0.3555', 

{'loss': '0.3752', 'grad_norm': '3.076', 'learning_rate': '9.597e-05', 'epoch': '4.029'}
{'loss': '0.399', 'grad_norm': '3.024', 'learning_rate': '9.588e-05', 'epoch': '4.121'}
{'loss': '0.3748', 'grad_norm': '2.822', 'learning_rate': '9.579e-05', 'epoch': '4.212'}
{'loss': '0.3574', 'grad_norm': '2.382', 'learning_rate': '9.57e-05', 'epoch': '4.304'}
{'loss': '0.3726', 'grad_norm': '3.116', 'learning_rate': '9.561e-05', 'epoch': '4.396'}
{'loss': '0.3825', 'grad_norm': '4.511', 'learning_rate': '9.551e-05', 'epoch': '4.487'}
{'loss': '0.3679', 'grad_norm': '3.127', 'learning_rate': '9.542e-05', 'epoch': '4.579'}
{'loss': '0.3669', 'grad_norm': '2.504', 'learning_rate': '9.533e-05', 'epoch': '4.67'}
{'loss': '0.354', 'grad_norm': '4.427', 'learning_rate': '9.524e-05', 'epoch': '4.762'}
{'loss': '0.3865', 'grad_norm': '2.94', 'learning_rate': '9.515e-05', 'epoch': '4.853'}
{'loss': '0.3253', 'grad_norm': '2.245', 'learning_rate': '9.506e-05', 'epoch': '4.945'}
{'eval_loss': '0.3293', 'e

...

{'loss': '0.09289', 'grad_norm': '1.526', 'learning_rate': '2.198e-05', 'epoch': '78.02'}
{'loss': '0.1107', 'grad_norm': '3.199', 'learning_rate': '2.189e-05', 'epoch': '78.11'}
{'loss': '0.1057', 'grad_norm': '0.6578', 'learning_rate': '2.18e-05', 'epoch': '78.21'}
{'loss': '0.09955', 'grad_norm': '0.7458', 'learning_rate': '2.171e-05', 'epoch': '78.3'}
{'loss': '0.09559', 'grad_norm': '0.5564', 'learning_rate': '2.161e-05', 'epoch': '78.39'}
{'loss': '0.1083', 'grad_norm': '2.194', 'learning_rate': '2.152e-05', 'epoch': '78.48'}
{'loss': '0.1223', 'grad_norm': '1.591', 'learning_rate': '2.143e-05', 'epoch': '78.57'}
{'loss': '0.1086', 'grad_norm': '0.2969', 'learning_rate': '2.134e-05', 'epoch': '78.66'}
{'loss': '0.1055', 'grad_norm': '0.7712', 'learning_rate': '2.125e-05', 'epoch': '78.75'}
{'loss': '0.1087', 'grad_norm': '1.365', 'learning_rate': '2.116e-05', 'epoch': '78.85'}
{'loss': '0.09898', 'grad_norm': '2.04', 'learning_rate': '2.106e-05', 'epoch': '78.94'}
{'eval_loss': '

{'loss': '0.1136', 'grad_norm': '1.81', 'learning_rate': '2.097e-05', 'epoch': '79.03'}
{'loss': '0.1034', 'grad_norm': '1.823', 'learning_rate': '2.088e-05', 'epoch': '79.12'}
{'loss': '0.1101', 'grad_norm': '0.6073', 'learning_rate': '2.079e-05', 'epoch': '79.21'}
{'loss': '0.1215', 'grad_norm': '1.514', 'learning_rate': '2.07e-05', 'epoch': '79.3'}
{'loss': '0.1044', 'grad_norm': '1.331', 'learning_rate': '2.061e-05', 'epoch': '79.4'}
{'loss': '0.1115', 'grad_norm': '0.7546', 'learning_rate': '2.051e-05', 'epoch': '79.49'}
{'loss': '0.09901', 'grad_norm': '2.116', 'learning_rate': '2.042e-05', 'epoch': '79.58'}
{'loss': '0.097', 'grad_norm': '0.629', 'learning_rate': '2.033e-05', 'epoch': '79.67'}
{'loss': '0.1009', 'grad_norm': '0.4042', 'learning_rate': '2.024e-05', 'epoch': '79.76'}
{'loss': '0.1009', 'grad_norm': '2.712', 'learning_rate': '2.015e-05', 'epoch': '79.85'}
{'loss': '0.1185', 'grad_norm': '1.607', 'learning_rate': '2.006e-05', 'epoch': '79.95'}
{'eval_loss': '0.1191'

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.decoder.weight', 'lm_head.decoder.bias'].


{'train_runtime': '1746', 'train_samples_per_second': '999.9', 'train_steps_per_second': '31.28', 'train_loss': '0.2064', 'epoch': '80'}


Training complete.
Best model saved to: /content/drive/MyDrive/ProjectRoot/checkpoints/hybrid_char_bpe/mlm15_L8_H512_A8_lr00001_ep100_setv70_m2/best_model
Trainer state saved to: /content/drive/MyDrive/ProjectRoot/checkpoints/hybrid_char_bpe/mlm15_L8_H512_A8_lr00001_ep100_setv70_m2/trainer_state.json


## Saving from Colab

This repository is public, so I can open it directly in Colab and save changes back through Colab's normal GitHub UI.

The training cell is configured to avoid widget-style progress output because that was what kept breaking the GitHub notebook preview after saving.
